# Gold Layer
This is where we model the data based on how the business wants the data organized and stored. 

Data will be read from the Silver Layer and transformed into the new data model.

Data will be written into the Gold Layer using Spark SQL. (Better for business users / data analysts)

![](/Workspace/Users/kristynguyen93@gmail.com/databricks_travel_warehouse/travel_diagram.png)

# Data Catalog for Gold Layer
---

### 1. **gold_travel.fact_flights**
- **Purpose:** Flight Information from January 2026 - June 2026
- **Columns:**

| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| flight_date | Flight date |
| airline | Airline Carrier |
| carrier_flight_number | Flight number |
| origin_iata_code | Origin airport IATA code |
| origin_city | Origin City |
| origin_state | Origin State |
| destination_iata_code | Destination airport IATA code |
| destination_city | Destination City |
| destination_state | Destination State |
| scheduled_departure_time | Scheduled departure time |
| actual_departure_time | Actual departure time |
| wheels_off | Time aircraft wheels take off |
| wheels_on | Time aircraft wheels touch down |
| scheduled_arrival_time | Scheduled arrival time |
| actual_arrival_time | Actual arrival time |
| cancelled | 1 indicates yes or true |
| cancellation_code | Reason for cancellation |
| diverted | 1 indicates yes or true |
| air_time | Flight time in minutes |
| distance | Distance between airports in miles |
| carriier_delay | Duration in minutes if there is a carrier delay |
| weather_delay | Duration in minutes if there is a weather delay |
| nas_delay | Duration in minutes if there is a National Air System Delay |
| security_delay | Duration in minutes if there is a security delay |
| late_aircraft_delay | Duration in minutes if there is a delay due to another aircraft |

---

### 2. **gold_travel.airlines**
- **Purpose:** Airline Information
- **Columns:**

| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| code | airline code |
| name | airline carrier name |

---

### 3. **gold_travel.dim_airports**
- **Purpose:** Airport Information
- **Columns:**

| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| iata_code | The three-letter IATA code for the airport (if it has one). |
| type | The type of the airport. Allowed values are "balloonport", "closed_airport", "heliport", "large_airport", "medium_airport", "seaplane_base", and "small_airport". |
| name | The official airport name, including "Airport", "Airstrip", etc. |
| latitude_deg | The airport latitude in decimal degrees (positive for north). |
| longitude_deg | The airport longitude in decimal degrees (positive for east). |
| continent | The code for the continent where the airport is (primarily) located. Allowed values are "AF" (Africa), "AN" (Antarctica), "AS" (Asia), "EU" (Europe), "NA" (North America), "OC" (Oceania), or "SA" (South America). |
| iso_country | The two-character ISO 3166:1-alpha2 code for the country where the airport is (primarily) located. A handful of unofficial, non-ISO codes are also in use, such as "XK" for Kosovo. |
| iso_region | An alphanumeric code for the high-level administrative subdivision of a country where the airport is primarily located (e.g. province, governorate), prefixed by the ISO2 country code and a hyphen. OurAirports uses ISO 3166:2 codes whenever possible, preferring higher administrative levels, but also includes some custom codes.  |
| scheduled_service | "yes" if the airport currently has scheduled airline service; "no" otherwise. |

---

### 4. **gold_travel.fact_weather**
- **Purpose:** Daily weather information from January 2026 - June 2026
- **Columns:**

| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| iata_code | The three-letter IATA code for the airport (if it has one). |
| date | The date of the daily aggregate in EST timezone. |
| temp_max | Highest air temperature of the day, measured 2 m above ground. |
| temp_min | Lowest air temperature of the day, measured 2 m above ground. |
| rain_sum | Total rain for the day from large-scale weather systems. |
| showers_sum | Total shower precipitation for the day (convective, short and intense bursts). |
| snowfall_sum | Total snowfall for the day. |
| precipitation_sum | Total of all precipitation for the day: rain, showers, and snow (as water equivalent). |
| wind_speed_max | 	Highest sustained wind speed of the day, measured 10 m above ground. |
| wind_gusts_max | Highest wind gust of the day, measured 10 m above ground. |


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Creating DataFrames for each silver table

df_flights = spark.table("silver_travel.flights")
df_airlines = spark.table("silver_travel.airlines")
df_airports = spark.table("silver_travel.airports")
df_weather = spark.table("silver_travel.weather")

# Creating gold.fact_flights

In [0]:
df_flights.display()

In [0]:
df_flights.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_travel.fact_flights")

In [0]:
%sql
SELECT *
FROM gold_travel.fact_flights
LIMIT 1000

# Creating gold_travel.dim_airlines

In [0]:
df_airlines.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_travel.dim_airlines")

In [0]:
%sql
SELECT *
FROM gold_travel.dim_airlines
LIMIT 1000

# Creating gold_travel.airports

In [0]:
df_airports = df_airports.select(
    df_airports["iata_code"],
    df_airports["type"],
    df_airports["name"],
    df_airports["latitude_deg"],
    df_airports["longitude_deg"],
    df_airports["continent"],
    df_airports["iso_country"],
    df_airports["iso_region"],
    df_airports["scheduled_service"]
)

In [0]:
df_airports.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_travel.dim_airports")

In [0]:
%sql
SELECT *
FROM gold_travel.dim_airports
LIMIT 1000

# Creating gold_travel.fact_weather

In [0]:
df_weather.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_travel.fact_weather")

In [0]:
%sql
SELECT *
FROM gold_travel.fact_weather
LIMIT 1000